<center><h1>BUILDING LLMS FOR PRODUCTION - CHAPTER 2</h1></center>

<p>Install the following packages first using the pip packages manager:</p>
<ul>
    <li>deeplake</li>
    <li>openai</li>
    <li>tiktoken</li>
    <li>transformers</li>
    <li>torch</li>
    <li>numpy</li>
    <li>deepseed</li>
    <li>trl</li>
    <li>peft</li>
    <li>wandb</li>
    <li>bitsandbytes</li>
    <li>accelerate</li>
    <li>tqdm</li>
    <li>neural_compressor</li>
    <li>omnx</li>
    <li>pandas</li>
    <li>scipy</li>
</ul>


In [ ]:
# %pip install deeplake
# %pip install openai
# %pip install tiktoken
# %pip install transformers
# %pip install torch
# %pip install deepseed
# %pip install trl
# %pip install peft
# %pip install wandb
# %pip install bitsandbytes
# %pip install accelerate
# %pip install tqdm
# %pip install neural_compressor
# %pip install onnx
# %pip install scipy

<h3>The architecture in action</h3>

In [1]:
!pip install -q transformers==4.53.0 bitsandbytes==0.46.0 accelerate==1.8.1


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\adity\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configure 8-bit quantization using the BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=False
)

In [4]:
OPT = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-1.3b",
    quantization_config=quantization_config,
    device_map="auto"
)

c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adity\.cache\huggingface\hub\models--facebook--opt-1.3b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 303.53it/s]


In [5]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-1.3b")

In [6]:
inp = "The quick brown fox jumps over the lazy dog"
inp_tokenized = tokenizer( inp, return_tensors="pt" )
print( inp_tokenized['input_ids'].size() )
print( inp_tokenized )

torch.Size([1, 10])
{'input_ids': tensor([[    2,   133,  2119,  6219, 23602, 13855,    81,     5, 22414,  2335]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [7]:
print( OPT.model )

OPTModel(
  (decoder): OPTDecoder(
    (embed_tokens): Embedding(50272, 2048, padding_idx=1)
    (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
    (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True, bias=True)
    (layers): ModuleList(
      (0-23): 24 x OPTDecoderLayer(
        (self_attn): OPTAttention(
          (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (out_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
        )
        (activation_fn): ReLU()
        (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True, bias=True)
        (fc1): Linear8bitLt(in_features=2048, out_features=8192, bias=True)
        (fc2): Linear8bitLt(in_features=8192, out_features=2048, bias=True)
        (final_layer_norm): LayerNorm

In [8]:
embedded_input = OPT.model.decoder.embed_tokens(inp_tokenized['input_ids'].to(OPT.device))
print( "Layer:\t", OPT.model.decoder.embed_tokens )
print( "Size:\t", embedded_input.size() )
print( "Output:\t", embedded_input )

Layer:	 Embedding(50272, 2048, padding_idx=1)
Size:	 torch.Size([1, 10, 2048])
Output:	 tensor([[[-0.0407,  0.0519,  0.0574,  ..., -0.0263, -0.0355, -0.0260],
         [-0.0371,  0.0220, -0.0096,  ...,  0.0265, -0.0166, -0.0030],
         [-0.0455, -0.0236, -0.0121,  ...,  0.0043, -0.0166,  0.0193],
         ...,
         [ 0.0007,  0.0267,  0.0257,  ...,  0.0622,  0.0421,  0.0279],
         [-0.0126,  0.0347, -0.0352,  ..., -0.0393, -0.0396, -0.0102],
         [-0.0115,  0.0319,  0.0274,  ..., -0.0472, -0.0059,  0.0341]]],
       dtype=torch.float16, grad_fn=<EmbeddingBackward0>)


In [9]:
embed_pos_input = OPT.model.decoder.embed_positions(inp_tokenized['attention_mask'].to(OPT.device))
print( "Layer:\t", OPT.model.decoder.embed_positions )
print( "Size:\t", embed_pos_input.size() )
print( "Output:\t", embed_pos_input )

Layer:	 OPTLearnedPositionalEmbedding(2050, 2048)
Size:	 torch.Size([1, 10, 2048])
Output:	 tensor([[[-8.1406e-03, -2.6221e-01,  6.0768e-03,  ...,  1.7273e-02,
          -5.0621e-03, -1.6220e-02],
         [-8.0585e-05,  2.5000e-01, -1.6632e-02,  ..., -1.5419e-02,
          -1.7838e-02,  2.4948e-02],
         [-9.9411e-03, -1.4978e-01,  1.7557e-03,  ...,  3.7117e-03,
          -1.6434e-02, -9.9087e-04],
         ...,
         [ 3.6979e-04, -7.7454e-02,  1.2955e-02,  ...,  3.9330e-03,
          -1.1642e-02,  7.8506e-03],
         [-2.6779e-03, -2.2446e-02, -1.6754e-02,  ..., -1.3142e-03,
          -7.8583e-03,  2.0096e-02],
         [-8.6288e-03,  1.4233e-01, -1.9012e-02,  ..., -1.8463e-02,
          -9.8572e-03,  8.7662e-03]]], dtype=torch.float16,
       grad_fn=<EmbeddingBackward0>)


In [11]:
embed_position_input = embedded_input + embed_pos_input
hidden_states, _ = OPT.model.decoder.layers[0].self_attn(embed_position_input)
print( "Layer:\t", OPT.model.decoder.layers[0].self_attn )
print( "Size:\t", hidden_states.size() )
print( "Output:\t", hidden_states )

Layer:	 OPTAttention(
  (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
  (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
  (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
  (out_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
)
Size:	 torch.Size([1, 10, 2048])
Output:	 tensor([[[-0.0136, -0.0095,  0.0012,  ...,  0.0067, -0.0018,  0.0131],
         [-0.0131, -0.0100,  0.0022,  ...,  0.0088,  0.0003,  0.0124],
         [-0.0131, -0.0061,  0.0038,  ...,  0.0099,  0.0021,  0.0141],
         ...,
         [-0.0121, -0.0099,  0.0051,  ...,  0.0095,  0.0016,  0.0098],
         [-0.0120, -0.0103,  0.0052,  ...,  0.0094,  0.0012,  0.0091],
         [-0.0119, -0.0110,  0.0056,  ...,  0.0095,  0.0013,  0.0093]]],
       dtype=torch.float16, grad_fn=<MatMul8bitLtBackward>)
